# [실습③] 비기능 테스트 — 응답시간 · 비용 · 견고성

2일차 4교시. p50/p95 응답시간, 오류율, 건당 비용을 측정하고 이상 입력에 대한 동작을 확인합니다.

In [ ]:
# 실습 저장소 내려받기 + API 키 등록
# API 키는 왼쪽 🔑(보안 비밀) 메뉴에 OPENAI_API_KEY 이름으로 등록해 둡니다. 코드에 직접 붙여 넣지 않습니다.
![ -d /content/ai-quality-lab ] || git clone -q https://github.com/leejaehee1/ai-quality-lab.git /content/ai-quality-lab

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

In [ ]:
# 공통 예제 서비스 실행 (같은 런타임 안에서 http://localhost:8000 으로 호출)
# 팀 서비스를 테스트하는 조는 이 셀을 건너뛰고, 아래 SERVICE_URL만 바꿉니다.
!pip install -q fastapi uvicorn openai
!cd /content/ai-quality-lab/sample-service && nohup uvicorn main:app --port 8000 > server.log 2>&1 &

import time, requests
for _ in range(30):
    try:
        print(requests.get("http://localhost:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    print("서버가 뜨지 않았습니다. server.log를 확인하세요:")
    print(open("/content/ai-quality-lab/sample-service/server.log").read())

In [ ]:
import requests

SERVICE_URL = "http://localhost:8000/chat"   # 팀 서비스면 주소를 바꿉니다

def ask(question: str) -> dict:
    """서비스에 질문하고 답변·근거·토큰 사용량을 돌려준다. 팀 API 형식에 맞게 고칩니다."""
    r = requests.post(SERVICE_URL, json={"message": question}, timeout=60)
    r.raise_for_status()
    data = r.json()
    return {"answer": data["answer"],
            "contexts": data.get("sources", []),
            "usage": data.get("usage")}

print(ask("얼마씩 몇 달 줘?")["answer"])

## 응답시간·오류율 (질문당 20회)

In [ ]:
import time, statistics

def measure(question: str, n: int = 20):
    latencies, errors = [], 0
    for _ in range(n):
        t0 = time.perf_counter()
        try:
            ask(question)
        except Exception as e:
            errors += 1
            print("오류:", type(e).__name__, e)
            continue
        latencies.append(time.perf_counter() - t0)
    if len(latencies) < 2:
        return {"오류율": f"{errors}/{n}"}
    q = statistics.quantiles(latencies, n=20)      # 5% 간격 분위수, q[18] = p95
    return {"질문": question,
            "p50(초)": round(statistics.median(latencies), 2),
            "p95(초)": round(q[18], 2),
            "최대(초)": round(max(latencies), 2),
            "오류율": f"{errors}/{n}"}

import pandas as pd
pd.DataFrame([
    measure("얼마씩 몇 달 줘?"),
    measure("지원 대상 조건이랑 신청 방법, 제출 서류까지 전부 자세히 설명해 줘"),
])

## 건당 비용

가격은 **강의 당일 모델 공급사의 공식 가격표**에서 확인해 넣습니다 (1M 토큰당 USD).

In [ ]:
PRICE_IN, PRICE_OUT = 0.0, 0.0   # 1M 토큰당 USD — 공식 가격표 값으로 바꾸세요
USD_KRW = 1400

out = ask("얼마씩 몇 달 줘?")
u = out["usage"]
if not u:
    print("서비스가 토큰 사용량(usage)을 돌려주지 않습니다.")
elif PRICE_IN == 0 and PRICE_OUT == 0:
    print("PRICE_IN, PRICE_OUT에 가격을 먼저 입력하세요. 토큰:", u)
else:
    cost = (u["prompt_tokens"] * PRICE_IN + u["completion_tokens"] * PRICE_OUT) / 1_000_000 * USD_KRW
    print(f"입력 {u['prompt_tokens']} 토큰, 출력 {u['completion_tokens']} 토큰, 약 {cost:.2f}원")
    print(f"하루 1,000건이면 월 약 {cost * 1000 * 30:,.0f}원")

## 견고성 테스트 (R-01 ~ R-04, R-07)

R-05(API 장애), R-06(지연)은 서비스 설정을 바꿔야 하므로 `day2/checklists/robustness-fallback.md`를 보고 테스트 환경에서 따로 합니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def probe(label, message):
    t0 = time.perf_counter()
    try:
        r = requests.post(SERVICE_URL, json={"message": message}, timeout=60)
        body = r.text[:150].replace("\n", " ")
        return {"ID": label, "상태코드": r.status_code, "시간(초)": round(time.perf_counter() - t0, 2), "응답 앞부분": body}
    except Exception as e:
        return {"ID": label, "상태코드": "예외", "시간(초)": round(time.perf_counter() - t0, 2), "응답 앞부분": str(e)[:150]}

results = [
    probe("R-01 빈 입력", "   "),
    probe("R-02 긴 입력", "지원금 " * 1700),
    probe("R-03 오타·줄임말", "쳥년 지원금 얼마임 몇달줌"),
    probe("R-04 다른 언어", "How much is the youth support grant?"),
]

with ThreadPoolExecutor(max_workers=10) as ex:   # R-07: 동시에 10건
    burst = list(ex.map(lambda i: probe(f"R-07 연속 #{i}", "얼마씩 몇 달 줘?"), range(10)))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(results + burst)

**과제**: 결과를 `day2/checklists/robustness-fallback.md` 표에 옮기고, 기대 동작과 다른 항목은 결함 보고서로 작성합니다.